# Figure 3

**Applying Mycol to improve *Fusarium oxysporum* spore germination counting efficiency**

- **Panels**  **a** workflow strip (seeded geometry) · **b, c** one spore field, raw and
  class-coloured · **d** true vs predicted count · **e** DenseNet confusion matrix · **f** counting time
- **Needs**  the `case_study_2/` session (**b, c, e**); `assets/` for `cs2_cellpose3_results.csv`
  (panel **d**, from `analyses/cs2_cellpose3_evaluation.ipynb`) and the raters' workbook (panel **f**)
- **Writes**  `output/Figure_3.svg` and its 300 dpi `.png`, the raster the manuscript embeds;
  intermediates in `output/panels/`
- **Kernel**  `mycol_colonies_env`, run top to bottom


In [ ]:
import re
import subprocess
from pathlib import Path

from PIL import Image

ASSETS = Path("assets")
OUTPUT = Path("output")
PANELS = OUTPUT / "panels"          # intermediates: the workflow strip and the data panels
REPO_ROOT = Path.cwd().parents[2]
PANELS.mkdir(parents=True, exist_ok=True)


def wrote(path):
    print(f"  wrote {path.name:38} {path.stat().st_size:>9,} B")


def rasterise(svg_path, png_path, dpi=300):
    """Render the finished SVG to PNG at `dpi` with the Chrome kaleido installs.

    Chrome is already a dependency - plotly's static export drives it. SVG user units
    are CSS px at 96 dpi, hence the device scale factor of dpi/96.
    """
    try:
        from choreographer.cli import _cli_utils
        chrome = str(_cli_utils.get_chrome_sync())
    except Exception:
        chrome = "/Applications/Google Chrome.app/Contents/MacOS/Google Chrome"
        if not Path(chrome).exists():
            raise RuntimeError("no Chrome - run `plotly_get_chrome`, or install Google Chrome")

    svg = svg_path.read_text()
    w, h = (round(float(v)) for v in
            re.search(r'viewBox="0 0 ([\d.]+) ([\d.]+)"', svg[:800]).groups())
    page = OUTPUT / "_render.html"
    page.write_text("<!doctype html><meta charset=utf-8><style>"
                    "html,body{margin:0;padding:0;background:#fff}svg{display:block}</style>"
                    + svg[svg.index("<svg"):])
    # Chrome reserves ~87 CSS px of window furniture even headless and clips the page
    # by that much, so render into a taller window and crop back.
    scale, raw = dpi / 96, OUTPUT / "_render.png"
    subprocess.run([chrome, "--headless", "--disable-gpu", "--hide-scrollbars",
                    f"--force-device-scale-factor={scale}", f"--window-size={w},{h + 200}",
                    f"--screenshot={raw.resolve()}", page.resolve().as_uri()],
                   check=True, capture_output=True)
    with Image.open(raw) as im:
        im.crop((0, 0, round(w * scale), round(h * scale))).save(png_path)
    raw.unlink()
    page.unlink()
    wrote(png_path)


## Panel a - the workflow strip

Five step panels plus the flow that composes them, all from seeded geometry, so the strip is
deterministic. The assembly cell inlines `figure3_flow.svg`.


In [ ]:
import math
import random

PREFIX = "figure3"
PANEL_W, PANEL_H = 240, 250
ART_X, ART_Y, ART_S = 30, 48, 180   # the 180x180 art square, in authoring coords
ART_SHIFT = -28                     # art sits high in the panel, title below it
TITLE_Y = 224                       # title baseline
GAP, PAD, STRIP_MARGIN = 52, 28, 16

STRIP_STYLE = """
    text     { font-family: -apple-system, BlinkMacSystemFont, "Segoe UI", Roboto,
               Helvetica, Arial, sans-serif; fill:#0f172a; }
    .title   { font-size:13px; font-weight:600; text-anchor:middle; }

    .panel   { fill:#ffffff; stroke:#cbd5e1; stroke-width:1.5; }
    .frame   { fill:#ffffff; stroke:#0f172a; stroke-width:1.2; }

    /* spores: raw, masked, and the two classes */
    .cell    { fill:#94a3b8; fill-opacity:0.30; stroke:#0f172a; stroke-width:0.9; }
    .mask    { fill:#e5431e; fill-opacity:0.17; stroke:#0f172a; stroke-width:2;
               stroke-linejoin:round; }
    .clsA    { fill:#5289C7; fill-opacity:0.45; stroke:#0f172a; stroke-width:2;
               stroke-linejoin:round; }
    .clsB    { fill:#4EB265; fill-opacity:0.45; stroke:#0f172a; stroke-width:2;
               stroke-linejoin:round; }
    .cursor  { fill:#0f172a; stroke:#ffffff; stroke-width:1.1; stroke-linejoin:round; }

    .err     { fill:none; stroke:#c2410c; stroke-width:1.6; stroke-dasharray:4 3.5; }
    .card    { fill:#ffffff; stroke:#0f172a; stroke-width:1.2; }
    .glyph   { fill:#94a3b8; fill-opacity:0.45; stroke:#0f172a; stroke-width:0.9; }
    .barA    { fill:#5289C7; fill-opacity:0.75; }
    .barB    { fill:#4EB265; fill-opacity:0.75; }
    .dl      { fill:none; stroke:#1d4ed8; stroke-width:3; stroke-linecap:round;
               stroke-linejoin:round; }
    .arrow   { fill:none; stroke:#334155; stroke-width:1.8; }
    .arrowh  { fill:#334155; }
"""

CURSOR = ('<path class="cursor" transform="translate({x},{y}) scale({s})" '
          'd="M 0,0 0,15 3.9,11.4 6.7,17 9.3,15.7 6.5,10.3 11.9,10.1 Z" />')


def harrow(x1, x2, y):
    return (f'<path class="arrow" d="M {x1},{y} H {x2 - 7}" />'
            f'<path class="arrowh" d="M {x2},{y} {x2 - 8},{y - 4.8} {x2 - 8},{y + 4.8} Z" />')


In [ ]:
# ── the spore field: ten spores, four of them germinated, each germ tube aimed
#    where it fits inside the frame and stays clear of its neighbours ──
def scatter_box(rng, x, y, w, h, n, min_dist):
    """n points inside a box, no two closer than min_dist."""
    pts, guard = [], 0
    while len(pts) < n and guard < 8000:
        guard += 1
        p = (rng.uniform(x, x + w), rng.uniform(y, y + h))
        if all(math.dist(p, q) >= min_dist for q in pts):
            pts.append(p)
    return pts


def fit_ray(x, y, ang, want, r, x0, y0, x1, y1):
    """Longest ray from (x,y) at `ang` keeping a disc of radius r inside the box."""
    dx, dy = math.cos(ang), math.sin(ang)
    lim = want
    for d, p, lo, hi in ((dx, x, x0 + r, x1 - r), (dy, y, y0 + r, y1 - r)):
        if d > 1e-9:
            lim = min(lim, (hi - p) / d)
        elif d < -1e-9:
            lim = min(lim, (lo - p) / d)
    return max(0.0, lim)


def aim_inside(rng, x, y, want, r, box, others):
    """Pick a direction whose full-length tube fits in the box and clears others.

    Tries evenly spaced headings, keeps those that fit, and among them takes the one
    whose tip ends up furthest from any neighbouring spore. If the point is too near a
    corner for any heading to fit, takes the roomiest one instead.
    """
    x0, y0, x1, y1 = box
    cands = [(a, fit_ray(x, y, a, want, r, x0, y0, x1, y1))
             for a in (2 * math.pi * i / 24 + rng.uniform(-0.06, 0.06) for i in range(24))]
    ok = [c for c in cands if c[1] >= want - 0.01] or [max(cands, key=lambda c: c[1])]

    def clearance(c):
        a, L = c
        tip = (x + L * math.cos(a), y + L * math.sin(a))
        mid = (x + L / 2 * math.cos(a), y + L / 2 * math.sin(a))
        return min((min(math.dist(tip, o), math.dist(mid, o)) for o in others), default=0.0)

    return max(ok, key=clearance)


def tapered_capsule(ax, ay, R, bx, by, r, n=22):
    """Outline of the convex hull of two circles - a tube with a bulbous end."""
    d = math.hypot(bx - ax, by - ay)
    if d < 1e-6:
        return f'M {ax + R},{ay} A {R},{R} 0 1 0 {ax - R},{ay} A {R},{R} 0 1 0 {ax + R},{ay} Z'
    th = math.atan2(by - ay, bx - ax)
    phi = math.asin(max(-1.0, min(1.0, (R - r) / d)))
    a1, a2 = th + math.pi / 2 + phi, th - math.pi / 2 - phi

    pts = [(ax + R * math.cos(a1), ay + R * math.sin(a1)),
           (bx + r * math.cos(a1), by + r * math.sin(a1))]
    for i in range(1, n + 1):                      # around the tip, through th
        t = a1 + (a2 - a1) * i / n
        pts.append((bx + r * math.cos(t), by + r * math.sin(t)))
    pts.append((ax + R * math.cos(a2), ay + R * math.sin(a2)))
    for i in range(1, n + 1):                      # around the bulb, the long way
        t = a2 + ((a1 - 2 * math.pi) - a2) * i / n
        pts.append((ax + R * math.cos(t), ay + R * math.sin(t)))
    return "M " + " L ".join(f"{x:.1f},{y:.1f}" for x, y in pts) + " Z"


_rng = random.Random(19)
_BOX = (ART_X + 3, ART_Y + 3, ART_X + ART_S - 3, ART_Y + ART_S - 3)
_pts = scatter_box(_rng, ART_X + 16, ART_Y + 16, ART_S - 32, ART_S - 32, 10, 45)

SPORES = []
for i, (x, y) in enumerate(_pts):
    R = _rng.uniform(8.0, 9.5)
    ang, length = aim_inside(_rng, x, y, _rng.uniform(28, 38), R * 0.55, _BOX,
                             [p for j, p in enumerate(_pts) if j != i])
    SPORES.append({"x": x, "y": y, "germ": i % 5 in (0, 2), "R": R,     # ~2 in 5 germinated
                   "tx": x + length * math.cos(ang), "ty": y + length * math.sin(ang)})

MISLABELLED = 4                                    # the classifier gets this one wrong


def spore_path(s):
    if s["germ"]:
        return tapered_capsule(s["x"], s["y"], s["R"], s["tx"], s["ty"], s["R"] * 0.55)
    return (f'M {s["x"] + s["R"]:.1f},{s["y"]:.1f} '
            f'A {s["R"]:.1f},{s["R"]:.1f} 0 1 0 {s["x"] - s["R"]:.1f},{s["y"]:.1f} '
            f'A {s["R"]:.1f},{s["R"]:.1f} 0 1 0 {s["x"] + s["R"]:.1f},{s["y"]:.1f} Z')


def field(cls_for):
    out = [f'<rect class="frame" x="{ART_X}" y="{ART_Y}" width="{ART_S}" height="{ART_S}" />']
    out += [f'<path class="{cls_for(i, s)}" d="{spore_path(s)}" />' for i, s in enumerate(SPORES)]
    return "\n  ".join(out)


def step3_classify():
    """Classified: germinated blue, ungerminated green - one call is wrong."""
    return field(lambda i, s: "clsA" if (not s["germ"] if i == MISLABELLED else s["germ"])
                 else "clsB")


def step4_curate():
    """The wrong call corrected by hand."""
    s = SPORES[MISLABELLED]
    return (field(lambda i, sp: "clsA" if sp["germ"] else "clsB") + "\n  "
            + f'<ellipse class="err" cx="{s["x"]:.1f}" cy="{s["y"]:.1f}" rx="21" ry="21" />'
            + "\n  " + CURSOR.format(x=s["x"] + 13, y=s["y"] + 7, s=1.0))


def step5_export():
    """Per-image counts, split by class, leaving as a file."""
    out = ['<rect class="card" x="42" y="60" width="156" height="96" rx="8" />']
    for i, (a, b) in enumerate([(46, 30), (28, 52), (60, 22)]):
        y = 82 + i * 26
        out.append(f'<rect class="glyph" x="56" y="{y - 6}" width="14" height="12" rx="2" />')
        out.append(f'<rect class="barA" x="78" y="{y - 5}" width="{a}" height="10" rx="5" />')
        out.append(f'<rect class="barB" x="{78 + a + 3}" y="{y - 5}" width="{b}" '
                   f'height="10" rx="5" />')
    out.append('<path class="dl" d="M 120,170 V 190" />')
    out.append('<path d="M 120,202 111,189 129,189 Z" fill="#1d4ed8" />')
    out.append('<path class="dl" d="M 96,204 V 212 H 144 V 204" />')
    return "\n  ".join(out)


STEPS = [("step1_spores", lambda: field(lambda i, s: "cell"), "Fungal spore images"),
         ("step2_segment", lambda: field(lambda i, s: "mask"), "Cellpose segmentation"),
         ("step3_classify", step3_classify, "DenseNet classification"),
         ("step4_curate", step4_curate, "Manual curation"),
         ("step5_export", step5_export, "Export class counts")]


In [ ]:
STEP_SVG = """<?xml version="1.0" encoding="UTF-8" standalone="no"?>
<svg width="{w}" height="{h}" viewBox="0 0 {w} {h}" version="1.1"
     xmlns="http://www.w3.org/2000/svg">
  <title>{title}</title>
  <style>{style}  </style>
  <rect class="panel" x="0.75" y="0.75" width="{iw}" height="{ih}" rx="10" />
  <g transform="translate(0,{shift})">
  {art}
  </g>
  <text class="title" x="{tx}" y="{ty}">{title}</text>
</svg>
"""

# one standalone SVG per step, and the same bodies composed into the flow
bodies = []
for name, fn, title in STEPS:
    svg = STEP_SVG.format(w=PANEL_W, h=PANEL_H, iw=PANEL_W - 1.5, ih=PANEL_H - 1.5,
                          style=STRIP_STYLE, art=fn(), title=title,
                          tx=PANEL_W / 2, ty=TITLE_Y, shift=ART_SHIFT)
    (PANELS / f"{PREFIX}_{name}.svg").write_text(svg)
    inner = re.sub(r"^.*?<svg[^>]*>", "", svg, flags=re.S).rsplit("</svg>", 1)[0]
    bodies.append(re.sub(r"<style>.*?</style>", "", inner, flags=re.S).strip())

n = len(STEPS)
band_w = PAD * 2 + n * PANEL_W + (n - 1) * GAP
W = STRIP_MARGIN * 2 + band_w
H = STRIP_MARGIN * 2 + PAD * 2 + PANEL_H
y = STRIP_MARGIN + PAD
xs = [STRIP_MARGIN + PAD + i * (PANEL_W + GAP) for i in range(n)]

parts = [f'<rect x="0" y="0" width="{W}" height="{H}" fill="#ffffff" />',
         f'<rect x="{STRIP_MARGIN}" y="{STRIP_MARGIN}" width="{band_w}" '
         f'height="{PAD * 2 + PANEL_H}" rx="18" fill="#f5f8ff" '
         f'stroke="#dbe6fb" stroke-width="1" />']
parts += [harrow(xs[i] + PANEL_W + 10, xs[i + 1] - 10, y + PANEL_H / 2) for i in range(n - 1)]
parts += [f'<g transform="translate({x},{y})">\n{body}\n  </g>'
          for x, body in zip(xs, bodies)]

(PANELS / f"{PREFIX}_flow.svg").write_text(
    f'<?xml version="1.0" encoding="UTF-8" standalone="no"?>\n'
    f'<!-- Generated by {PREFIX}.ipynb - edit that notebook, not this file. -->\n'
    f'<svg width="{W}" height="{H}" viewBox="0 0 {W} {H}" version="1.1"\n'
    f'     xmlns="http://www.w3.org/2000/svg">\n'
    f'  <style>{STRIP_STYLE}  </style>\n' + "\n  ".join(parts) + "\n</svg>\n")

print(f"  {n} step SVGs + {PREFIX}_flow.svg ({W}x{H})")
wrote(PANELS / f"{PREFIX}_flow.svg")


## Panels b to f - the data

Agg is forced - otherwise matplotlib picks the macOS backend, applies 2x Retina scaling and snaps
figure sizes to whole device pixels, shifting a panel by a couple of pixels.


In [ ]:
import json
import zipfile

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import ndimage as ndi

SESSION_DIR = REPO_ROOT / "case_study_2" / "mycol_saved_session_CS2"
if not SESSION_DIR.is_dir():
    print(f"  unpacking {SESSION_DIR.name}.zip (first run only)")
    zipfile.ZipFile(SESSION_DIR.with_suffix(".zip")).extractall(SESSION_DIR)

CP3_CSV = ASSETS / "cs2_cellpose3_results.csv"              # Cellpose 3 test-set counts
TIME_XLSX = ASSETS / "20250908_All_data_manual_vs_tool.xlsx"

# How the app names mask files: newer sessions write <stem>_masks.tif, older ones <stem>.tif.
# Read it off the session rather than assuming, so a re-export cannot silently break panel c.
MASK_SUFFIX = json.loads((SESSION_DIR / "image_metadata.json").read_text()).get("mask_suffix", "")

for p in (SESSION_DIR, CP3_CSV, TIME_XLSX):
    shown = p.relative_to(REPO_ROOT) if p.is_absolute() else p
    print(f"{'ok     ' if p.exists() else 'MISSING'}  {shown}")
print(f"mask suffix: {MASK_SUFFIX!r}")


In [ ]:
# --- the layout the assembly cell below uses, and the type size it implies ---
FIG_W, MARGIN, GAP_X = 1788, 20, 26
COL_W = (FIG_W - 2 * MARGIN - 2 * GAP_X) / 3        # 565.33 px - one column
WIDE_W = 2 * COL_W + GAP_X                          # 1156.67 px - panel f spans two
TEXT_PX = 19.5                                      # every label in the figure, in px

PANEL_IN = 4.5                                      # square panels, inches
WIDE_IN = PANEL_IN * WIDE_W / COL_W                 # same shown-scale, so same type size
DPI = 300
FONT_PT = TEXT_PX / ((DPI / 72) * (COL_W / (PANEL_IN * DPI)))

plt.rcParams.update({"font.size": FONT_PT, "axes.labelsize": FONT_PT,
                     "xtick.labelsize": FONT_PT, "ytick.labelsize": FONT_PT,
                     "legend.fontsize": FONT_PT})


def save_panel(fig, stem):
    """Write a panel as PNG (what the figure embeds) and SVG (for editing)."""
    for ext in ("png", "svg"):
        fig.savefig(PANELS / f"{stem}.{ext}", dpi=DPI, facecolor="white")
    px = Image.open(PANELS / f"{stem}.png").size
    print(f"  wrote {stem}.png / .svg  ({px[0]}x{px[1]} px)")


print(f"column {COL_W:.1f} px, wide panel {WIDE_W:.1f} px")
print(f"FONT_PT = {FONT_PT:.2f}  ->  {TEXT_PX} px in the finished figure")


In [ ]:
# --- panels b and c: one spore field, raw and with class-coloured masks ---
IMAGE = "ICI Dkmt1 4-0070 IMAGEJ"
CLASS_COLOURS = {"Germinated": "#5289C7", "Ungerminated": "#4EB265"}   # as the workflow strip
UNCLASSIFIED = "#94a3b8"
FILL_ALPHA = 0.45
SCALE = 2                       # store at 2x native so mask outlines survive rasterisation
OUTLINE_PX = 4                  # at SCALE; ~2 px once shown at COL_W

raw = Image.open(SESSION_DIR / "images" / f"{IMAGE}.tif").convert("RGB")
labels = np.array(Image.open(SESSION_DIR / "masks" / f"{IMAGE}{MASK_SUFFIX}.tif"))
rows = pd.read_csv(SESSION_DIR / "cell_metrics.csv").query(f"image == '{IMAGE}.tif'")

# the CSV's "mask #" is claimed to be the label value in the TIFF - verify, don't assume
ids = [int(i) for i in np.unique(labels) if i > 0]
assert ids == sorted(int(v) for v in rows["mask #"]), "cell_metrics rows do not match mask labels"
label_class = dict(zip(rows["mask #"].astype(int), rows["mask label"]))

# upscale: LANCZOS for the photograph, NEAREST for the labels so no class bleeds into another
big = raw.resize((raw.width * SCALE, raw.height * SCALE), Image.LANCZOS)
labels_big = np.array(Image.fromarray(labels).resize(big.size, Image.NEAREST))

overlay = np.asarray(big, dtype=float).copy()
for i in ids:
    colour = np.array([int(CLASS_COLOURS.get(label_class[i], UNCLASSIFIED)[j:j + 2], 16)
                       for j in (1, 3, 5)], dtype=float)
    m = labels_big == i
    overlay[m] = (1 - FILL_ALPHA) * overlay[m] + FILL_ALPHA * colour
    overlay[m & ~ndi.binary_erosion(m, iterations=OUTLINE_PX)] = colour

big.save(PANELS / "spore_raw.png")
Image.fromarray(overlay.astype("uint8")).save(PANELS / "spore_overlay.png")

in_test = IMAGE in set(pd.read_csv(CP3_CSV).query('split == "test"')["image"])
print(f"{IMAGE}  native {raw.size[0]}x{raw.size[1]} -> stored {big.size[0]}x{big.size[1]}")
print("  " + ", ".join(f"{k} {v}" for k, v in sorted(rows["mask label"].value_counts().items())))
print(f"  in panel d's held-out split: {in_test}")


In [ ]:
# --- panel d: true vs predicted count on the held-out test images ---
SCATTER_COLOUR = "#D55E00"       # Okabe-Ito vermillion; clear of the two class colours

test = pd.read_csv(CP3_CSV).query('split == "test"')


def mape(true, pred):
    """Mean absolute percentage error, the metric used across the case studies."""
    true, pred = np.asarray(true, float), np.asarray(pred, float)
    return float((np.abs(pred - true) / true * 100).mean())


fig, ax = plt.subplots(figsize=(PANEL_IN, PANEL_IN), layout="constrained")
true, pred = test["true_count"].to_numpy(float), test["pred_count"].to_numpy(float)
lim = max(true.max(), pred.max()) * 1.05

ax.plot([0, lim], [0, lim], color="#8a8a8a", lw=1.2, ls="--", zorder=1)   # perfect counter
ax.scatter(true, pred, s=40, alpha=0.7, color=SCATTER_COLOUR, zorder=2,
           edgecolors="white", linewidths=0.5)
ax.set_xlim(-lim * 0.02, lim)
ax.set_ylim(-lim * 0.02, lim)
ax.set_aspect("equal")
ax.text(0.05, 0.95, f"MAPE {mape(true, pred):.2f}%\nn = {len(true):,}", transform=ax.transAxes,
        va="top", ha="left", fontsize=FONT_PT,
        bbox=dict(boxstyle="round,pad=0.35", fc="white", ec="#dddddd"))
ax.text(lim * 0.97, lim * 0.9, "y = x", color="#8a8a8a", fontsize=FONT_PT, rotation=45,
        rotation_mode="anchor", va="bottom", ha="right")
ax.set_xlabel("True Count", fontsize=FONT_PT)
ax.set_ylabel("Predicted Count", fontsize=FONT_PT)
ax.grid(True, lw=0.4, color="#ededed")
ax.set_axisbelow(True)
for sp in ("top", "right"):
    ax.spines[sp].set_visible(False)
save_panel(fig, "scatter_counts")

print(f"  {len(test)} test images, {test['true_count'].sum():,} cells, "
      f"MAPE {mape(true, pred):.2f}%")


In [ ]:
# --- panel e: DenseNet confusion matrix, straight out of the saved session ---
trace = json.loads((SESSION_DIR / "densenet_confusion_matrix.json").read_text())["data"][0]
cm = np.array([[int(v) for v in row] for row in trace["text"]])   # rows = true, cols = predicted
classes = list(trace["x"])
assert list(trace["y"]) == classes and trace["type"] == "heatmap"

row_share = cm / cm.sum(axis=1, keepdims=True)
accuracy = np.trace(cm) / cm.sum()

fig, ax = plt.subplots(figsize=(PANEL_IN, PANEL_IN), layout="constrained")
ax.imshow(row_share, cmap="Blues", vmin=0, vmax=1)
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        ax.text(j, i, f"{cm[i, j]:,}", ha="center", va="center", fontsize=FONT_PT,
                color="white" if row_share[i, j] > 0.5 else "#0f172a")
ax.set_xticks(range(len(classes)), classes)
ax.set_yticks(range(len(classes)), classes, rotation=90, va="center")
ax.set_xlabel("Predicted Class", fontsize=FONT_PT)
ax.set_ylabel("True Class", fontsize=FONT_PT)
ax.tick_params(length=0)
for sp in ax.spines.values():
    sp.set_visible(False)
ax.text(0, 1.04, f"Accuracy {accuracy * 100:.1f}%   n = {cm.sum():,}",
        transform=ax.transAxes, ha="left", va="bottom", fontsize=FONT_PT)
save_panel(fig, "germination_confusion_matrix")

print(f"  {cm.sum():,} held-out patches, accuracy {accuracy * 100:.1f}%")
for i, c in enumerate(classes):
    print(f"    {c:13s} recall {row_share[i, i] * 100:5.1f}%  ({cm[i, i]:,} of {cm[i].sum():,})")


In [ ]:
# --- panel f: counting time, manual vs mycol-assisted ---
from scipy import stats

# Manual takes a pale sky blue; the two mycol components share a purple ramp. No green here -
# panel c spends green on ungerminated cells. This tint is far lighter and less saturated than
# panel c's germinated blue (#5289C7), so the two do not read as the same encoding.
C_MANUAL, C_COUNT, C_PROC = "#bcdcee", "#b8a9d9", "#6e5fa6"

t = (pd.read_excel(TIME_XLSX, sheet_name="time")
       .rename(columns={"Unnamed: 0": "User", "Manual ": "Manual"}))
t = t[t["User"].isin(["User1", "User2", "User3"])].copy()
t["count_pct"] = t["Mycol Assisted"] / t["Manual"] * 100
t["proc_pct"] = t["Processing Time"] / t["Manual"] * 100

total = (t["count_pct"] + t["proc_pct"]).values
t_stat, p_val = stats.ttest_rel(np.full(len(total), 100.0), total)
sd = float(np.std(total, ddof=1))

labels = list(t["User"]) + ["Mean"]
counting = list(t["count_pct"]) + [t["count_pct"].mean()]
processing = list(t["proc_pct"]) + [t["proc_pct"].mean()]

x = np.arange(len(labels))
w = 0.36
fig, ax = plt.subplots(figsize=(WIDE_IN, PANEL_IN), layout="constrained")
bar = dict(edgecolor="black", linewidth=1)
ax.bar(x - w / 2, [100.0] * 4, w, color=C_MANUAL, label="Manual", **bar)
ax.bar(x + w / 2, counting, w, color=C_COUNT, label="Mycol counting", **bar)
ax.bar(x + w / 2, processing, w, bottom=counting, color=C_PROC, label="Mycol processing",
       yerr=[0, 0, 0, sd], error_kw=dict(ecolor="black", elinewidth=1, capsize=5), **bar)

# significance bracket over the Mean group, clear of the error bar
y_top = max(100.0, counting[-1] + processing[-1] + sd) + 8
star = "***" if p_val < 0.001 else "**" if p_val < 0.01 else "*" if p_val < 0.05 else "ns"
ax.plot([x[-1] - w / 2, x[-1] + w / 2], [y_top, y_top], color="black", lw=1)
ax.text(x[-1], y_top + 1.5, star, ha="center", va="bottom", fontsize=FONT_PT)

ax.set_xticks(x, labels)
ax.set_ylim(0, y_top + 22)
ax.set_ylabel("% of Manual Counting Time", fontsize=FONT_PT)
ax.legend(loc="upper left", frameon=True, framealpha=0.85, edgecolor="#dddddd",
          fontsize=FONT_PT, handlelength=1.1, handletextpad=0.5, borderpad=0.4,
          columnspacing=1.0, ncols=3)
ax.tick_params(axis="x", length=0)
for sp in ("top", "right"):
    ax.spines[sp].set_visible(False)
save_panel(fig, "time_comparison_panel")

print(f"  mycol total {total.mean():.1f}% of manual (SD {sd:.1f}) "
      f"-> {100 - total.mean():.1f}% less time")
print(f"  paired t-test n={len(total)}: t = {t_stat:.3f}, p = {p_val:.4f}  ->  {star}")


## Assembly

Lays panel **a** and the panels out on the 1788 px canvas, draws the panel letters (here and nowhere
else), and embeds the rasters as data URIs so the `.svg` stands alone. The `.svg` is the master; the
`.png` the manuscript embeds is rendered from it at 300 dpi.


In [ ]:
import base64

OUT = OUTPUT / "Figure_3.svg"

ROW_GAP = 26
LETTER_H, LETTER_SIZE = 36, 30      # strip above each row for its panel letter
CONTENT_W = FIG_W - 2 * MARGIN

FLOW_SVG = PANELS / f"{PREFIX}_flow.svg"
FLOW_MARGIN = 16            # the flow's own gutter, backed out so its band spans the figure

# row 2: the scatter fills its column; the two micrographs are squared off to the
# scatter's plot area instead, so all three read as one height
ROW2 = [{"letter": "b", "name": "raw field", "src": PANELS / "spore_raw.png", "micrograph": True},
        {"letter": "c", "name": "classified field", "src": PANELS / "spore_overlay.png",
         "micrograph": True},
        {"letter": "d", "name": "count scatter", "src": PANELS / "scatter_counts.png"}]

# row 3: the confusion matrix, then the timing comparison across the other two columns
ROW3 = [{"letter": "e", "name": "confusion matrix", "w": COL_W,
         "src": PANELS / "germination_confusion_matrix.png"},
        {"letter": "f", "name": "counting time", "w": WIDE_W,
         "src": PANELS / "time_comparison_panel.png"}]

# panel c's legend: a swatch per class. Top-right is the emptiest corner of this
# particular field - change the image and check it again.
LEGEND_CORNER = "top-right"
SWATCH, LEGEND_PAD, LEGEND_INSET, LINE_H = 20, 12, 14, 28
CHAR_W = 0.52               # rough advance width per character, in units of TEXT_PX

STYLE = """
    text { font-family: -apple-system, BlinkMacSystemFont, "Segoe UI", Roboto,
           Helvetica, Arial, sans-serif; fill:#0f172a; }
    .fig-letter { font-size:%dpx; font-weight:700; }
    .legend { font-size:%.1fpx; }
""" % (LETTER_SIZE, TEXT_PX)


def data_uri(path):
    return f"data:image/png;base64,{base64.b64encode(path.read_bytes()).decode()}"


def plot_area(svg_path):
    """Where a matplotlib panel's plot area sits inside its square, as fractions.

    Measured rather than assumed: matplotlib writes the axes background as the first
    path inside <g id="axes_1">, so its corners are the plot rectangle to the pixel.
    Returns (x0, y0, width, height) in 0-1 units, y running down as it does in SVG.
    """
    txt = svg_path.read_text()
    cw, ch = (float(v) for v in
              re.search(r'viewBox="0 0 ([\d.]+) ([\d.]+)"', txt[:2000]).groups())
    patch = re.search(r'<path d="([^"]+)"', txt[txt.find('id="axes_1"'):])
    nums = [float(v) for v in re.findall(r"-?[\d.]+", patch.group(1))]
    xs, ys = nums[0::2], nums[1::2]
    return (min(xs) / cw, min(ys) / ch, (max(xs) - min(xs)) / cw, (max(ys) - min(ys)) / ch)


def inline_flow(x, y, scale):
    """Drop the workflow SVG in as vector, keeping its own <style>.

    `x`/`y` place the flow's *band*, not its canvas: FLOW_MARGIN is backed out so the
    band lines up edge to edge with the panels below. The flow's 13 px titles would land
    at 13*scale px, so they are rewritten to hit TEXT_PX exactly; only this inlined copy
    is touched, and the standalone flow SVG keeps its own sizing.
    """
    inner = re.sub(r"^.*?<svg[^>]*>", "", FLOW_SVG.read_text(), flags=re.S).rsplit("</svg>", 1)[0]
    # drop the flow's white backdrop: pulled flush to the figure edge it would reach up
    # over the panel letter and hide it
    inner = re.sub(r'<rect x="0" y="0" width="\d+" height="\d+" fill="#ffffff" ?/>',
                   "", inner, count=1)
    inner = re.sub(r"(\.title\s*\{[^}]*?font-size:)[\d.]+px",
                   lambda m: f"{m.group(1)}{TEXT_PX / scale:.2f}px", inner, count=1)
    ox, oy = x - FLOW_MARGIN * scale, y - FLOW_MARGIN * scale
    return f'<g transform="translate({ox:.2f},{oy:.2f}) scale({scale:.5f})">{inner}</g>'


def legend(x, y, size, colours, corner):
    """Class key for panel c, drawn as vector so its labels come out at TEXT_PX like
    everything else rather than baked into the micrograph at whatever size that raster
    happens to be shown at. `x`/`y`/`size` describe the panel it sits inside."""
    names = list(colours)
    box_w = 2 * LEGEND_PAD + SWATCH + 10 + max(len(n) for n in names) * TEXT_PX * CHAR_W
    box_h = 2 * LEGEND_PAD + len(names) * LINE_H - (LINE_H - SWATCH)
    bx = x + LEGEND_INSET if corner.endswith("left") else x + size - LEGEND_INSET - box_w
    by = y + LEGEND_INSET if corner.startswith("top") else y + size - LEGEND_INSET - box_h

    parts = [f'<rect x="{bx:.1f}" y="{by:.1f}" width="{box_w:.1f}" height="{box_h:.1f}" '
             f'rx="8" fill="#ffffff" fill-opacity="0.86" stroke="#cbd5e1" stroke-width="1" />']
    for i, name in enumerate(names):
        sy = by + LEGEND_PAD + i * LINE_H
        parts.append(f'<rect x="{bx + LEGEND_PAD:.1f}" y="{sy:.1f}" width="{SWATCH}" '
                     f'height="{SWATCH}" rx="4" fill="{colours[name]}" fill-opacity="0.85" '
                     f'stroke="#0f172a" stroke-width="1.4" />')
        parts.append(f'<text class="legend" x="{bx + LEGEND_PAD + SWATCH + 10:.1f}" '
                     f'y="{sy + SWATCH * 0.82:.1f}">{name}</text>')
    return "\n  ".join(parts)


In [ ]:
print(f"figure width {FIG_W}, column {COL_W:.0f} px, wide panel {WIDE_W:.0f} px")

# the scatter's plot area sets the height every panel in its row is cut to
_px0, py0, _pw, ph = plot_area(ROW2[2]["src"].with_suffix(".svg"))
micro_side = COL_W * ph                  # the micrographs match the plot area's height
micro_dy = COL_W * py0                   # and start level with its top edge
print(f"  row align      plot area y0={py0:.4f} h={ph:.4f} -> panels b/c "
      f"{micro_side:.0f} px square, {micro_dy:.0f} px down from the row top")

for spec in ROW2 + ROW3:
    im = Image.open(spec["src"])
    shown = micro_side if spec.get("micrograph") else spec.get("w", COL_W)
    print(f"  panel {spec['letter']}  {spec['name']:<18} {spec['src'].name:<32} "
          f"{im.width}x{im.height} -> {shown:.0f} px wide")

# scale on the band, not the canvas, so panel a spans the same width as b-f
flow_w, flow_canvas_h = (float(v) for v in
                         re.search(r'viewBox="0 0 ([\d.]+) ([\d.]+)"',
                                   FLOW_SVG.read_text()[:600]).groups())
flow_scale = CONTENT_W / (flow_w - 2 * FLOW_MARGIN)
flow_h = (flow_canvas_h - 2 * FLOW_MARGIN) * flow_scale
print(f"  panel a  {'workflow':<18} {FLOW_SVG.name:<32} {flow_w:.0f}x{flow_canvas_h:.0f} -> "
      f"band {CONTENT_W:.0f} px ({flow_scale:.2f}x)")

# panel f is drawn at exactly the aspect that makes it as tall as a column, so both
# lower rows are COL_W high and the grid stays square
row_h = COL_W
y_a = MARGIN + LETTER_H
y_b = y_a + flow_h + ROW_GAP + LETTER_H
y_e = y_b + row_h + ROW_GAP + LETTER_H
fig_h = int(round(y_e + row_h + MARGIN))

parts = [f'<rect x="0" y="0" width="{FIG_W}" height="{fig_h}" fill="#ffffff" />',
         f'<text class="fig-letter" x="{MARGIN}" y="{MARGIN + LETTER_SIZE}">a</text>',
         inline_flow(MARGIN, y_a, flow_scale)]

for i, spec in enumerate(ROW2):
    x = MARGIN + i * (COL_W + GAP_X)
    # the letter stays on the column edge; only the picture is cut to the plot area
    parts.append(f'<text class="fig-letter" x="{x:.1f}" y="{y_b - LETTER_H + LETTER_SIZE:.1f}">'
                 f'{spec["letter"]}</text>')
    side, top = (micro_side, y_b + micro_dy) if spec.get("micrograph") else (COL_W, y_b)
    parts.append(f'<image x="{x:.1f}" y="{top:.1f}" width="{side:.2f}" height="{side:.2f}" '
                 f'href="{data_uri(spec["src"])}" />')
    if spec["letter"] == "c":
        parts.append(legend(x, top, side, CLASS_COLOURS, LEGEND_CORNER))

x = MARGIN
for spec in ROW3:
    parts.append(f'<text class="fig-letter" x="{x:.1f}" y="{y_e - LETTER_H + LETTER_SIZE:.1f}">'
                 f'{spec["letter"]}</text>')
    parts.append(f'<image x="{x:.1f}" y="{y_e:.1f}" width="{spec["w"]:.2f}" '
                 f'height="{row_h:.2f}" href="{data_uri(spec["src"])}" />')
    x += spec["w"] + GAP_X

OUT.write_text(
    f'<?xml version="1.0" encoding="UTF-8" standalone="no"?>\n'
    f'<!-- Generated by {PREFIX}.ipynb - edit that notebook, not this file. -->\n'
    f'<svg width="{FIG_W}" height="{fig_h}" viewBox="0 0 {FIG_W} {fig_h}"\n'
    f'     version="1.1" xmlns="http://www.w3.org/2000/svg"\n'
    f'     xmlns:xlink="http://www.w3.org/1999/xlink">\n'
    f'  <title>Figure 3 - spore germination</title>\n'
    f'  <style>{STYLE}  </style>\n  ' + "\n  ".join(parts) + "\n</svg>\n")

print()
wrote(OUT)
rasterise(OUT, OUTPUT / "Figure_3.png")


## What this notebook wrote


In [ ]:
from IPython.display import display

for p in sorted(OUTPUT.rglob("*")):
    if p.is_file():
        print(f"  {str(p.relative_to(OUTPUT)):44} {p.stat().st_size:>9,} B")

im = Image.open(OUTPUT / "Figure_3.png")
print(f"\nFigure 3   {im.width} x {im.height} px")
im.thumbnail((760, 760))
display(im.convert("RGB"))
